<a href="https://colab.research.google.com/github/sadumina/Deep-Learning-Assignment-Group-ID-5/blob/model%2FDenseNet121/train_densenet121_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Diabetic Retinopathy Classification — DenseNet121 (Transfer Learning)

**Dataset:** APTOS 2019 Blindness Detection, pre-processed version *Diabetic Retinopathy 224x224 Gaussian Filtered* (Kaggle, 3,662 fundus images).
**Task:** 5-class severity grading — No DR, Mild, Moderate, Severe, Proliferative DR.
**Approach:** DenseNet121 pre-trained on ImageNet, trained in two stages:
1. **Feature extraction** — backbone frozen, only a new classification head is trained.
2. **Fine-tuning** — the last 60 backbone layers are unfrozen and trained with a very small learning rate.

**Fair comparison:** the train/validation split is identical to the Classic-CNN model (80/20, `seed=42`).

> Before running: **Runtime → Change runtime type → T4 GPU**

## 1. Environment setup
Mount Google Drive (to save results permanently) and import the libraries.

In [1]:
# Mount Google Drive so checkpoints and plots survive a Colab disconnect
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# Confirm a GPU is available (training on CPU would be very slow)
print("TensorFlow version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

Mounted at /content/drive
TensorFlow version: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration
All key settings in one place. `SEED`, `IMG_SIZE` and the 20% validation split must stay the same as the Classic-CNN notebook so both models see the same images.

In [2]:
SEED = 42                 # same seed as Classic-CNN -> same train/val split
IMG_SIZE = (224, 224)     # DenseNet121's native input size
BATCH_SIZE = 32
VAL_SPLIT = 0.2           # 80% train / 20% validation

STAGE1_EPOCHS = 10        # head training (frozen backbone)
STAGE2_EPOCHS = 20        # fine-tuning
STAGE1_LR = 1e-3
STAGE2_LR = 1e-5          # small LR so pre-trained weights are not destroyed
UNFREEZE_LAST = 60        # number of DenseNet121 layers to unfreeze in stage 2

# All outputs (models, plots) are saved here on Google Drive
OUT_DIR = "/content/drive/MyDrive/DL Assignment/Data/DR_DenseNet121_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)   # reproducible weights, shuffling and augmentation

## 3. Load the dataset
The dataset is stored in Google Drive as a zip file (`My Drive/DL Assignment/Data.zip`).
It is unzipped to Colab's local disk, which is much faster than reading thousands of small files from Drive.

In [3]:
# ---- Load data from Google Drive (zip file) ----
# My Drive > DL Assignment > Data.zip
ZIP_PATH = "/content/drive/MyDrive/DL Assignment/Data.zip"

# Unzip to Colab's local disk (fast, and avoids incomplete folder uploads)
DATA_ROOT = "/content/data"
if os.path.isdir(DATA_ROOT):
    shutil.rmtree(DATA_ROOT)          # remove any previous incomplete copy
print("Unzipping dataset...")
shutil.unpack_archive(ZIP_PATH, DATA_ROOT)
print("Done.")

# Find the folder that directly contains the 5 class folders (works however the zip is nested)
DATA_DIR = next(d for d, sub, _ in os.walk(DATA_ROOT) if "No_DR" in sub and "Mild" in sub)
print("DATA_DIR:", DATA_DIR)

# Check image counts per class (must total 3662)
total = 0
for cls in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, cls)
    if os.path.isdir(p):
        n = len(os.listdir(p)); total += n
        print(f"{cls:15s} {n}")
print("Total:", total)
assert total == 3662, "Dataset incomplete"

Unzipping dataset...
Done.
DATA_DIR: /content/data/gaussian_filtered_images/gaussian_filtered_images
Mild            370
Moderate        999
No_DR           1805
Proliferate_DR  295
Severe          193
Total: 3662
